# Paper 173 Section 7.2 Implementation: Camera FPS

Static Section 7.2 DAG/list-scheduling implementation using four ResNet IDK classifiers with a configurable camera FPS cap.


## Imports

This cell imports the libraries used by the notebook. The worker import happens in the constants cell because it needs the project path first.


In [1]:
import json
import sys
import tarfile
import time
from collections import Counter
from itertools import combinations, permutations
from pathlib import Path

import numpy as np
import torch
import torch.multiprocessing as mp
from PIL import Image
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms


## Constants

Edit this cell to change models, devices, paths, threshold policy, sample counts, processor count, camera FPS, and output files. Runtime worker groups are selected later by the Section 7 DAG/list scheduler.


In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RUNS_DIR = PROJECT_ROOT / "runs"
IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"
RUNS_DIR.mkdir(exist_ok=True)

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}

MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet50"
MODEL_D = "resnet152"

MODEL_LABELS = ("A", "B", "C", "D")
MODEL_BY_LABEL = {
    "A": MODEL_A,
    "B": MODEL_B,
    "C": MODEL_C,
    "D": MODEL_D,
}
LABEL_BY_MODEL = {model_name: label for label, model_name in MODEL_BY_LABEL.items()}
MODELS = tuple(MODEL_BY_LABEL[label] for label in MODEL_LABELS)

MODEL_A_DEVICE = "mps"
MODEL_B_DEVICE = "mps"
MODEL_C_DEVICE = "mps"
MODEL_D_DEVICE = "mps"
MODEL_DEVICES = {
    MODEL_A: MODEL_A_DEVICE,
    MODEL_B: MODEL_B_DEVICE,
    MODEL_C: MODEL_C_DEVICE,
    MODEL_D: MODEL_D_DEVICE,
}

MPS_WORKER_COUNT = 4
WORKER_NAMES = tuple(f"mps-worker-{index}" for index in range(MPS_WORKER_COUNT))
WORKER_INDEX_BY_NAME = {worker_name: index for index, worker_name in enumerate(WORKER_NAMES)}

PAPER_CONFIDENCE_THRESHOLDS = {
    MODEL_A: 0.890,
    MODEL_B: 0.895,
    MODEL_C: 0.896,
    MODEL_D: 0.910,
}
RECOMPUTE_CONFIDENCE_THRESHOLDS = True
PRECISION_THRESHOLD = 0.95

PROFILE_PREFIXES = ("matched", "top")
TEST_VARIANT = "threshold-0.7"
MAX_SAMPLES = 10000
CAMERA_FPS = 60.0  # Positive FPS cap; use None for unlimited input
BATCH_SIZE = 1
CLASSIFICATION_THRESHOLD = None
LATENCY_CONSTRAINT_MS = float("inf")
LATENCY_EXECUTION_TIME_PERCENTILE = 95
SAVE_RESULTS = True
RESULTS_PATH = RUNS_DIR / "paper173_sec7_camera_fps_results.json"
PREDICTIONS_PATH = RUNS_DIR / "paper173_sec7_camera_fps_predictions.npz"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from real_time_mp_workers import model_group_worker


## Image Transform

This cell defines the ImageNet preprocessing used for the real-time test images.


In [3]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


## Dataset Loader

This cell reads ImageNetV2 rows from the local tar archives, applies the ImageNet transform, and returns a DataLoader for the real-time test.


In [4]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_imagenet_v2_rows(variant, max_samples):
    emitted = 0
    with tarfile.open(VARIANT_ARCHIVES[variant], "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            if image_file is None:
                continue
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield image, label_from_key(member.name)
            emitted += 1
            if emitted >= max_samples:
                break


class StreamingImageNetV2Dataset(IterableDataset):
    def __init__(self, variant, max_samples):
        self.variant = variant
        self.max_samples = int(max_samples)

    def __iter__(self):
        for image, label in stream_imagenet_v2_rows(self.variant, self.max_samples):
            yield preprocess(image), int(label)

    def __len__(self):
        return self.max_samples


def make_streaming_loader(variant, max_samples):
    dataset = StreamingImageNetV2Dataset(variant, max_samples)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
    )
    return dataset, loader


## Profiling Tables

This cell builds the Section 7.2 profiling inputs from cached logits: calibrated IDK thresholds, empirical arbitrary-dependence success probabilities, and cached execution-time summaries.


In [5]:
def load_cache(prefix, model_name):
    path = ARTIFACTS_DIR / f"{prefix}_{model_name}.npz"
    with np.load(path) as data:
        return {
            "probabilities": data["probabilities"],
            "predictions": data["predictions"],
            "labels": data["labels"],
            "times_ms": data["times_ms"],
        }


def cache_confidence(cache):
    return np.asarray(cache["probabilities"]).max(axis=1)


def label_subsets(labels):
    for subset_size in range(len(labels) + 1):
        for subset in combinations(labels, subset_size):
            yield frozenset(subset)


def load_profile_data(model_name):
    confidence_parts = []
    correct_parts = []
    time_parts = []

    for prefix in PROFILE_PREFIXES:
        cache = load_cache(prefix, model_name)
        confidence_parts.append(cache_confidence(cache).astype(np.float64))
        correct_parts.append(cache["predictions"] == cache["labels"])
        time_parts.append(np.asarray(cache["times_ms"], dtype=np.float64))

    return {
        "confidence": np.concatenate(confidence_parts),
        "correct": np.concatenate(correct_parts),
        "times_ms": np.concatenate(time_parts),
    }


def threshold_for_precision(confidences, correct, precision_threshold):
    order = np.argsort(-confidences)
    sorted_confidences = confidences[order]
    sorted_correct = correct[order]
    tie_ends = np.r_[
        np.flatnonzero(sorted_confidences[:-1] != sorted_confidences[1:]),
        len(sorted_confidences) - 1,
    ]
    precision = np.cumsum(sorted_correct)[tie_ends] / (tie_ends + 1)
    valid = np.flatnonzero(precision >= precision_threshold)
    if len(valid) == 0:
        raise ValueError(f"No threshold reaches precision >= {precision_threshold}")
    return float(sorted_confidences[tie_ends[valid[-1]]])


def threshold_metrics(confidences, correct, threshold):
    non_idk = confidences >= threshold
    return {
        "non_idk_rate": float(non_idk.mean()),
        "precision_on_non_idk": float(correct[non_idk].mean()) if non_idk.any() else 0.0,
    }


def build_thresholds(profile_data_by_label):
    thresholds = {}
    rows = []
    for label in MODEL_LABELS:
        model_name = MODEL_BY_LABEL[label]
        data = profile_data_by_label[label]
        if RECOMPUTE_CONFIDENCE_THRESHOLDS:
            threshold = threshold_for_precision(
                data["confidence"],
                data["correct"],
                PRECISION_THRESHOLD,
            )
        else:
            threshold = PAPER_CONFIDENCE_THRESHOLDS[model_name]

        thresholds[model_name] = threshold
        metrics = threshold_metrics(data["confidence"], data["correct"], threshold)
        rows.append(
            {
                "label": label,
                "model": model_name,
                "threshold": float(threshold),
                "non_idk_rate": metrics["non_idk_rate"],
                "precision_on_non_idk": metrics["precision_on_non_idk"],
            }
        )
    return thresholds, rows


def build_success_probability_table(success_matrix):
    probability_by_subset = {frozenset(): 0.0}
    for subset in label_subsets(MODEL_LABELS):
        if not subset:
            continue
        column_indexes = [MODEL_LABELS.index(label) for label in subset]
        probability_by_subset[subset] = float(success_matrix[:, column_indexes].any(axis=1).mean())
    return probability_by_subset


def subset_name(subset):
    return "".join(label for label in MODEL_LABELS if label in subset) or "empty"


PROFILE_DATA_BY_LABEL = {
    label: load_profile_data(MODEL_BY_LABEL[label])
    for label in MODEL_LABELS
}
CONFIDENCE_THRESHOLDS, THRESHOLD_VALIDATION_TABLE = build_thresholds(PROFILE_DATA_BY_LABEL)

profile_success_matrix = np.column_stack(
    [
        PROFILE_DATA_BY_LABEL[label]["confidence"] >= CONFIDENCE_THRESHOLDS[MODEL_BY_LABEL[label]]
        for label in MODEL_LABELS
    ]
)
SUCCESS_PROBABILITY_BY_SUBSET = build_success_probability_table(profile_success_matrix)
SUCCESS_PROBABILITY_TABLE = [
    {"subset": subset_name(subset), "probability": float(probability)}
    for subset, probability in sorted(
        SUCCESS_PROBABILITY_BY_SUBSET.items(),
        key=lambda item: (len(item[0]), subset_name(item[0])),
    )
]

MEAN_EXECUTION_TIME_MS_BY_LABEL = {
    label: float(PROFILE_DATA_BY_LABEL[label]["times_ms"].mean())
    for label in MODEL_LABELS
}
LATENCY_EXECUTION_TIME_MS_BY_LABEL = {
    label: float(np.percentile(PROFILE_DATA_BY_LABEL[label]["times_ms"], LATENCY_EXECUTION_TIME_PERCENTILE))
    for label in MODEL_LABELS
}
MAX_EXECUTION_TIME_MS_BY_LABEL = {
    label: float(PROFILE_DATA_BY_LABEL[label]["times_ms"].max())
    for label in MODEL_LABELS
}
MEAN_TIMING_TABLE = [
    {
        "label": label,
        "model": MODEL_BY_LABEL[label],
        "mean_ms": MEAN_EXECUTION_TIME_MS_BY_LABEL[label],
    }
    for label in MODEL_LABELS
]
LATENCY_TIMING_TABLE = [
    {
        "label": label,
        "model": MODEL_BY_LABEL[label],
        "p95_ms": LATENCY_EXECUTION_TIME_MS_BY_LABEL[label],
        "max_ms": MAX_EXECUTION_TIME_MS_BY_LABEL[label],
    }
    for label in MODEL_LABELS
]

FULL_SUCCESS_PROBABILITY = SUCCESS_PROBABILITY_BY_SUBSET[frozenset(MODEL_LABELS)]
CLASSIFICATION_THRESHOLD_VALUE = (
    FULL_SUCCESS_PROBABILITY
    if CLASSIFICATION_THRESHOLD is None
    else float(CLASSIFICATION_THRESHOLD)
)

print("Profiling threshold table")
print("label model threshold non_idk_rate precision")
for row in THRESHOLD_VALIDATION_TABLE:
    print(
        f"{row['label']} {row['model']} "
        f"{row['threshold']:.6f} {row['non_idk_rate']:.4f} {row['precision_on_non_idk']:.4f}"
    )

print()
print("Timing table")
print("label model mean_ms p95_ms max_ms")
for label in MODEL_LABELS:
    print(
        f"{label} {MODEL_BY_LABEL[label]} "
        f"{MEAN_EXECUTION_TIME_MS_BY_LABEL[label]:.3f} "
        f"{LATENCY_EXECUTION_TIME_MS_BY_LABEL[label]:.3f} "
        f"{MAX_EXECUTION_TIME_MS_BY_LABEL[label]:.3f}"
    )

print()
print("Profile rows:", len(profile_success_matrix))
print("Classification threshold:", round(CLASSIFICATION_THRESHOLD_VALUE, 6))
print("Full-set success probability:", round(FULL_SUCCESS_PROBABILITY, 6))


Profiling threshold table
label model threshold non_idk_rate precision
A resnet18 0.915247 0.3505 0.9501
B resnet34 0.911178 0.4217 0.9502
C resnet50 0.688761 0.0389 0.9511
D resnet152 0.769386 0.4382 0.9500

Timing table
label model mean_ms p95_ms max_ms
A resnet18 3.136 3.383 18.481
B resnet34 5.311 5.813 7.721
C resnet50 10.736 11.134 16.295
D resnet152 24.313 25.715 34.442

Profile rows: 20000
Classification threshold: 0.58975
Full-set success probability: 0.58975


## Section 7.2 DAG

This cell implements the Section 7 multiprocessor DAG/list scheduler. The expected-duration cost uses mean finish events, while latency pruning uses cached p95 timing.


In [6]:
COST_TOLERANCE = 1e-9


class DagVertex:
    def __init__(self, completed_sets, running_labels):
        self.completed_sets = tuple(frozenset(labels) for labels in completed_sets)
        self.running_labels = tuple(running_labels)
        self.key = (
            tuple(tuple(sorted(labels)) for labels in self.completed_sets),
            self.running_labels,
        )
        self.mean_makespans = self._makespans(MEAN_EXECUTION_TIME_MS_BY_LABEL)
        self.latency_makespans = self._makespans(LATENCY_EXECUTION_TIME_MS_BY_LABEL)
        self.selected_worker = self._selected_worker()
        self.finish_time_ms = self.mean_makespans[self.selected_worker] if self.selected_worker is not None else 0.0
        self.success_set = self._success_set()
        self.success_probability = SUCCESS_PROBABILITY_BY_SUBSET[frozenset(self.success_set)]
        self.latency_stop_time_ms = self._latency_stop_time()
        self.cost_ms = float("inf")
        self.previous_key = None
        self.added_labels = ()

    def _makespans(self, time_by_label):
        makespans = []
        for completed, running in zip(self.completed_sets, self.running_labels):
            total = sum(time_by_label[label] for label in completed)
            if running is not None:
                total += time_by_label[running]
            makespans.append(float(total))
        return tuple(makespans)

    def _selected_worker(self):
        active_workers = [index for index, label in enumerate(self.running_labels) if label is not None]
        if not active_workers:
            return None
        return min(active_workers, key=lambda index: (self.mean_makespans[index], index))

    def _success_set(self):
        labels = set()
        for completed in self.completed_sets:
            labels.update(completed)
        if self.selected_worker is None:
            return frozenset(labels)

        for index, running in enumerate(self.running_labels):
            if running is not None and abs(self.mean_makespans[index] - self.finish_time_ms) <= COST_TOLERANCE:
                labels.add(running)
        return frozenset(labels)

    def _latency_stop_time(self):
        if not self.success_set:
            return 0.0

        stop_times = []
        for index, completed in enumerate(self.completed_sets):
            total = sum(LATENCY_EXECUTION_TIME_MS_BY_LABEL[label] for label in completed)
            running = self.running_labels[index]
            if running in self.success_set and running not in completed:
                total += LATENCY_EXECUTION_TIME_MS_BY_LABEL[running]
            if total:
                stop_times.append(total)
        return float(max(stop_times))

    def used_labels(self):
        labels = set()
        for completed in self.completed_sets:
            labels.update(completed)
        labels.update(label for label in self.running_labels if label is not None)
        return frozenset(labels)

    def running_count(self):
        return sum(label is not None for label in self.running_labels)


def make_vertex(completed_sets, running_labels):
    return DagVertex(completed_sets, running_labels)


def move_selected(vertex, next_label):
    completed_sets = [set(labels) for labels in vertex.completed_sets]
    running_labels = list(vertex.running_labels)
    worker_index = vertex.selected_worker
    completed_sets[worker_index].add(running_labels[worker_index])
    running_labels[worker_index] = next_label
    return make_vertex(completed_sets, running_labels)


def outgoing_vertices(vertex):
    unused_labels = [label for label in MODEL_LABELS if label not in vertex.used_labels()]

    if vertex.running_count() == 0:
        max_start = min(MPS_WORKER_COUNT, len(unused_labels))
        for start_count in range(1, max_start + 1):
            for first_labels in permutations(unused_labels, start_count):
                running_labels = list(first_labels) + [None] * (MPS_WORKER_COUNT - start_count)
                yield make_vertex([set() for _ in WORKER_NAMES], running_labels), tuple(first_labels)
        return

    if unused_labels:
        for label in unused_labels:
            yield move_selected(vertex, label), (label,)
        return

    if vertex.running_count() > 1:
        yield move_selected(vertex, None), ()


def better_dag_vertex(candidate, current_best):
    if current_best is None:
        return True
    if candidate.cost_ms < current_best.cost_ms - COST_TOLERANCE:
        return True
    return (
        abs(candidate.cost_ms - current_best.cost_ms) <= COST_TOLERANCE
        and len(candidate.used_labels()) < len(current_best.used_labels())
    )


def build_dag_solution():
    start = make_vertex([set() for _ in WORKER_NAMES], [None for _ in WORKER_NAMES])
    start.cost_ms = 0.0
    vertices = {start.key: start}
    frontier = [start]
    best_key = None

    while frontier:
        next_frontier_by_key = {}
        for vertex in frontier:
            if vertex.latency_stop_time_ms > LATENCY_CONSTRAINT_MS:
                continue
            if vertex.success_probability >= CLASSIFICATION_THRESHOLD_VALUE:
                best_vertex = vertices[best_key] if best_key is not None else None
                if better_dag_vertex(vertex, best_vertex):
                    best_key = vertex.key
                continue

            for next_vertex, added_labels in outgoing_vertices(vertex):
                if next_vertex.latency_stop_time_ms > LATENCY_CONSTRAINT_MS:
                    continue
                edge_ms = (next_vertex.finish_time_ms - vertex.finish_time_ms) * (1.0 - vertex.success_probability)
                cost_ms = vertex.cost_ms + edge_ms
                existing = vertices.get(next_vertex.key)
                if existing is None:
                    existing = next_vertex
                    vertices[existing.key] = existing
                if cost_ms < existing.cost_ms - COST_TOLERANCE:
                    existing.cost_ms = cost_ms
                    existing.previous_key = vertex.key
                    existing.added_labels = added_labels
                    next_frontier_by_key[existing.key] = existing
        frontier = list(next_frontier_by_key.values())

    if best_key is None:
        raise ValueError("No DAG vertex meets the classification threshold and latency constraint")
    return vertices, best_key


def recover_order(vertices, key):
    pieces = []
    while key is not None:
        vertex = vertices[key]
        if vertex.added_labels:
            pieces.append(vertex.added_labels)
        key = vertex.previous_key
    order = []
    for piece in reversed(pieces):
        order.extend(piece)
    return tuple(order)


def list_schedule(order_labels):
    groups = [[] for _ in WORKER_NAMES]
    mean_makespans = [0.0 for _ in WORKER_NAMES]
    finish_events = []

    for label in order_labels:
        worker_index = min(range(len(WORKER_NAMES)), key=lambda index: (mean_makespans[index], index))
        finish_ms = mean_makespans[worker_index] + MEAN_EXECUTION_TIME_MS_BY_LABEL[label]
        mean_makespans[worker_index] = finish_ms
        groups[worker_index].append(label)
        finish_events.append((finish_ms, label, worker_index))

    return tuple(tuple(group) for group in groups), tuple(sorted(finish_events)), float(max(mean_makespans, default=0.0))


def label_finish_times(groups, time_by_label):
    finish_times = {}
    for group in groups:
        elapsed_ms = 0.0
        for label in group:
            elapsed_ms += time_by_label[label]
            finish_times[label] = float(elapsed_ms)
    return finish_times


def evaluate_order(order_labels):
    groups, finish_events, mean_duration_ms = list_schedule(order_labels)
    latency_finish_times = label_finish_times(groups, LATENCY_EXECUTION_TIME_MS_BY_LABEL)
    latency_duration_ms = max(latency_finish_times.values(), default=0.0)
    cost_ms = 0.0
    previous_finish_ms = 0.0
    success_set = set()
    stop_time_ms = 0.0
    latency_stop_time_ms = 0.0
    stop_probability = 0.0
    event_index = 0

    while event_index < len(finish_events):
        finish_ms = finish_events[event_index][0]
        probability = SUCCESS_PROBABILITY_BY_SUBSET[frozenset(success_set)]
        cost_ms += (finish_ms - previous_finish_ms) * (1.0 - probability)

        while event_index < len(finish_events) and abs(finish_events[event_index][0] - finish_ms) <= COST_TOLERANCE:
            success_set.add(finish_events[event_index][1])
            event_index += 1

        previous_finish_ms = finish_ms
        stop_time_ms = finish_ms
        latency_stop_time_ms = max(latency_finish_times[label] for label in success_set)
        stop_probability = SUCCESS_PROBABILITY_BY_SUBSET[frozenset(success_set)]
        if stop_probability >= CLASSIFICATION_THRESHOLD_VALUE:
            break

    return {
        "order_labels": tuple(order_labels),
        "worker_groups_labels": groups,
        "expected_duration_ms": float(cost_ms),
        "mean_duration_ms": float(mean_duration_ms),
        "latency_duration_ms": float(latency_duration_ms),
        "stop_time_ms": float(stop_time_ms),
        "latency_stop_time_ms": float(latency_stop_time_ms),
        "success_probability": float(stop_probability),
    }


def better_order(candidate, current_best):
    if current_best is None:
        return True
    if candidate["expected_duration_ms"] < current_best["expected_duration_ms"] - COST_TOLERANCE:
        return True
    return (
        abs(candidate["expected_duration_ms"] - current_best["expected_duration_ms"]) <= COST_TOLERANCE
        and len(candidate["order_labels"]) < len(current_best["order_labels"])
    )


def exhaustive_search():
    best = None
    for order_size in range(1, len(MODEL_LABELS) + 1):
        for order_labels in permutations(MODEL_LABELS, order_size):
            result = evaluate_order(order_labels)
            if result["success_probability"] < CLASSIFICATION_THRESHOLD_VALUE:
                continue
            if result["latency_stop_time_ms"] > LATENCY_CONSTRAINT_MS:
                continue
            if better_order(result, best):
                best = result

    if best is None:
        raise ValueError("No exhaustive order meets the classification threshold and latency constraint")
    return best


DAG_VERTICES, DAG_BEST_KEY = build_dag_solution()
DAG_BEST_VERTEX = DAG_VERTICES[DAG_BEST_KEY]
DAG_OPTIMAL_ORDER_LABELS = recover_order(DAG_VERTICES, DAG_BEST_KEY)
DAG_OPTIMAL_RESULTS = evaluate_order(DAG_OPTIMAL_ORDER_LABELS)
DAG_OPTIMAL_RESULTS["expected_duration_ms"] = float(DAG_BEST_VERTEX.cost_ms)
DAG_OPTIMAL_RESULTS["stop_time_ms"] = float(DAG_BEST_VERTEX.finish_time_ms)
DAG_OPTIMAL_RESULTS["latency_stop_time_ms"] = float(DAG_BEST_VERTEX.latency_stop_time_ms)
DAG_OPTIMAL_RESULTS["success_probability"] = float(DAG_BEST_VERTEX.success_probability)
DAG_OPTIMAL_RESULTS["dag_vertex_count"] = len(DAG_VERTICES)

EXHAUSTIVE_VERIFIER_RESULT = exhaustive_search()
DAG_EXHAUSTIVE_MATCH = abs(
    DAG_OPTIMAL_RESULTS["expected_duration_ms"] - EXHAUSTIVE_VERIFIER_RESULT["expected_duration_ms"]
) <= 1e-6

print("Section 7 DAG simulation")
print("processor_count optimal_order worker_groups expected_duration_ms stop_time_ms success_probability dag_vertices")
print(
    MPS_WORKER_COUNT,
    DAG_OPTIMAL_ORDER_LABELS,
    DAG_OPTIMAL_RESULTS["worker_groups_labels"],
    round(DAG_OPTIMAL_RESULTS["expected_duration_ms"], 3),
    round(DAG_OPTIMAL_RESULTS["stop_time_ms"], 3),
    round(DAG_OPTIMAL_RESULTS["success_probability"], 6),
    len(DAG_VERTICES),
)

print()
print("Exhaustive verifier result")
print("optimal_order worker_groups expected_duration_ms success_probability match_status")
print(
    EXHAUSTIVE_VERIFIER_RESULT["order_labels"],
    EXHAUSTIVE_VERIFIER_RESULT["worker_groups_labels"],
    round(EXHAUSTIVE_VERIFIER_RESULT["expected_duration_ms"], 3),
    round(EXHAUSTIVE_VERIFIER_RESULT["success_probability"], 6),
    "DAG matches exhaustive search" if DAG_EXHAUSTIVE_MATCH else "DAG differs from exhaustive search",
)


Section 7 DAG simulation
processor_count optimal_order worker_groups expected_duration_ms stop_time_ms success_probability dag_vertices
4 ('A', 'B', 'C', 'D') (('A',), ('B',), ('C',), ('D',)) 14.471 24.313 0.58975 273

Exhaustive verifier result
optimal_order worker_groups expected_duration_ms success_probability match_status
('A', 'B', 'C', 'D') (('A',), ('B',), ('C',), ('D',)) 14.471 0.58975 DAG matches exhaustive search


## DAG-Selected Runtime Schedule

This cell converts the Section 7 DAG/list-schedule result into the static worker groups used by the shared-MPS runtime test.


In [7]:
RUNTIME_ORDER_LABELS = DAG_OPTIMAL_ORDER_LABELS
RUNTIME_CASCADE_ORDER = tuple(MODEL_BY_LABEL[label] for label in RUNTIME_ORDER_LABELS)
RUNTIME_SCHEDULE_RESULTS = DAG_OPTIMAL_RESULTS

WORKER_MODEL_GROUPS = tuple(
    tuple(MODEL_BY_LABEL[label] for label in group)
    for group in DAG_OPTIMAL_RESULTS["worker_groups_labels"]
)
ACTIVE_WORKER_NAMES = tuple(
    worker_name
    for worker_name, group in zip(WORKER_NAMES, WORKER_MODEL_GROUPS)
    if group
)
ACTIVE_WORKER_MODEL_GROUPS = tuple(group for group in WORKER_MODEL_GROUPS if group)
ACTIVE_WORKER_GROUP_BY_NAME = dict(zip(ACTIVE_WORKER_NAMES, ACTIVE_WORKER_MODEL_GROUPS))

print("Selected static schedule from Section 7 DAG/list scheduling")
print("Runtime order:", RUNTIME_ORDER_LABELS)
print("Runtime cascade models:", RUNTIME_CASCADE_ORDER)
print("Worker groups:")
for worker_name, group in zip(WORKER_NAMES, WORKER_MODEL_GROUPS):
    labels = tuple(LABEL_BY_MODEL[model_name] for model_name in group)
    print(f"  {worker_name}: {labels} {group}")
print("Expected duration (ms):", round(RUNTIME_SCHEDULE_RESULTS["expected_duration_ms"], 3))
print("Stop time (ms):", round(RUNTIME_SCHEDULE_RESULTS["stop_time_ms"], 3))
print("Success probability:", round(RUNTIME_SCHEDULE_RESULTS["success_probability"], 6))


Selected static schedule from Section 7 DAG/list scheduling
Runtime order: ('A', 'B', 'C', 'D')
Runtime cascade models: ('resnet18', 'resnet34', 'resnet50', 'resnet152')
Worker groups:
  mps-worker-0: ('A',) ('resnet18',)
  mps-worker-1: ('B',) ('resnet34',)
  mps-worker-2: ('C',) ('resnet50',)
  mps-worker-3: ('D',) ('resnet152',)
Expected duration (ms): 14.471
Stop time (ms): 24.313
Success probability: 0.58975


## Test Dataset

This cell creates the local ImageNetV2 test loader and prints the camera FPS cap and selected shared-MPS worker groups.


In [8]:
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for the shared-MPS runtime test")

real_test_dataset, real_test_loader = make_streaming_loader(TEST_VARIANT, MAX_SAMPLES)
real_sample_count = len(real_test_dataset)

print("Shared-MPS runtime test of the selected schedule")
print("Local samples:", real_sample_count)
print("Camera FPS cap:", CAMERA_FPS)
print("Dataset:", VARIANT_ARCHIVES[TEST_VARIANT])
print("Configured worker slots:", MPS_WORKER_COUNT)
print("Active shared-MPS workers:", len(ACTIVE_WORKER_NAMES))
for worker_name, group in zip(WORKER_NAMES, WORKER_MODEL_GROUPS):
    labels = tuple(LABEL_BY_MODEL[model_name] for model_name in group)
    print(f"  {worker_name}: {labels} {group}")


Shared-MPS runtime test of the selected schedule
Local samples: 10000
Camera FPS cap: 60.0
Dataset: /Users/abhinavgupta/dynamic-idk-cascades/ImageNet-V2 DataSet/imagenetv2-threshold0.7.tar.gz
Configured worker slots: 4
Active shared-MPS workers: 4
  mps-worker-0: ('A',) ('resnet18',)
  mps-worker-1: ('B',) ('resnet34',)
  mps-worker-2: ('C',) ('resnet50',)
  mps-worker-3: ('D',) ('resnet152',)


## Real-Time Run

This cell admits frames on an absolute camera clock, runs the selected static Section 7 schedule on shared MPS worker processes, and records classification, latency, and worker metrics.


In [9]:
def run_real_time_test(input_fps=CAMERA_FPS):
    if input_fps is not None and input_fps <= 0:
        raise ValueError("input_fps must be positive or None")
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    worker_queues = {worker_name: ctx.SimpleQueue() for worker_name in ACTIVE_WORKER_NAMES}
    result_queue = ctx.SimpleQueue()
    processes = [
        ctx.Process(
            target=model_group_worker,
            args=(worker_name, ACTIVE_WORKER_GROUP_BY_NAME[worker_name], "mps", worker_queues[worker_name], result_queue),
        )
        for worker_name in ACTIVE_WORKER_NAMES
    ]

    for process in processes:
        process.start()

    try:
        total_samples = len(real_test_dataset)
        labels = np.full(total_samples, -1, dtype=np.int64)
        final_predictions = np.full(total_samples, -1, dtype=np.int64)
        chosen_models = np.full(total_samples, "", dtype="<U32")
        success_flags = np.zeros(total_samples, dtype=bool)
        latencies_ms = np.full(total_samples, np.nan, dtype=np.float64)

        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        idk_count_by_model = Counter()
        run_start = time.perf_counter()
        frame_period = None if input_fps is None else 1.0 / input_fps

        for sample_index, (images, batch_labels) in enumerate(real_test_loader):
            if frame_period is not None:
                wait_seconds = run_start + sample_index * frame_period - time.perf_counter()
                if wait_seconds > 0:
                    time.sleep(wait_seconds)
            labels[sample_index] = int(batch_labels.item())
            sample_start = time.perf_counter()
            worker_positions = {worker_name: 0 for worker_name in ACTIVE_WORKER_NAMES}
            active_workers = set()
            sample_done = False
            for worker_name, group in ACTIVE_WORKER_GROUP_BY_NAME.items():
                model_name = group[0]
                worker_queues[worker_name].put((sample_index, model_name, images))
                active_workers.add(worker_name)
                execution_count_by_model[model_name] += 1

            while active_workers:
                message = result_queue.get()
                if message[0] == "error":
                    _, worker_name, error = message
                    raise RuntimeError(f"{worker_name} worker failed: {error}")

                result_sample_index, worker_name, model_name, probabilities, prediction, confidence, elapsed_ms = message
                if result_sample_index != sample_index:
                    raise RuntimeError(f"Unexpected sample index {result_sample_index}; expected {sample_index}")

                active_workers.remove(worker_name)
                execution_time_ms_by_model[model_name] += elapsed_ms

                if confidence < CONFIDENCE_THRESHOLDS[model_name]:
                    idk_count_by_model[model_name] += 1
                if not sample_done and confidence >= CONFIDENCE_THRESHOLDS[model_name]:
                    final_predictions[sample_index] = prediction
                    chosen_models[sample_index] = model_name
                    success_flags[sample_index] = True
                    latencies_ms[sample_index] = (time.perf_counter() - sample_start) * 1000.0
                    sample_done = True

                if not sample_done:
                    worker_positions[worker_name] += 1
                    group = ACTIVE_WORKER_GROUP_BY_NAME[worker_name]
                    if worker_positions[worker_name] < len(group):
                        next_model_name = group[worker_positions[worker_name]]
                        worker_queues[worker_name].put((sample_index, next_model_name, images))
                        active_workers.add(worker_name)
                        execution_count_by_model[next_model_name] += 1

            if not sample_done:
                latencies_ms[sample_index] = (time.perf_counter() - sample_start) * 1000.0

        total_wall_time_seconds = time.perf_counter() - run_start
        success_count = int(np.count_nonzero(success_flags))
        correct_predictions = int(np.count_nonzero(final_predictions == labels))
        correct_successes = int(np.count_nonzero((final_predictions == labels) & success_flags))
        total_execution_time_ms_by_model = {model_name: float(execution_time_ms_by_model[model_name]) for model_name in MODELS}
        mean_execution_time_ms_by_model = {
            model_name: float(execution_time_ms_by_model[model_name] / execution_count_by_model[model_name])
            if execution_count_by_model[model_name]
            else 0.0
            for model_name in MODELS
        }
        runtime_metrics = {
            "total_samples": total_samples,
            "success_count": success_count,
            "idk_count": int(total_samples - success_count),
            "successful_classification_rate": float(success_count / total_samples),
            "accuracy": float(correct_predictions / total_samples),
            "accuracy_on_successes": float(correct_successes / success_count) if success_count else 0.0,
            "mean_latency_ms": float(latencies_ms.mean()),
            "p95_latency_ms": float(np.percentile(latencies_ms, 95)),
            "throughput_fps": float(total_samples / total_wall_time_seconds),
            "configured_input_fps": None if input_fps is None else float(input_fps),
            "frame_period_ms": None if input_fps is None else float(frame_period * 1000.0),
        }

        results = {
            **runtime_metrics,
            "correct_predictions": correct_predictions,
            "correct_successes": correct_successes,
            "total_wall_time_seconds": float(total_wall_time_seconds),
            "p50_latency_ms": float(np.percentile(latencies_ms, 50)),
            "runtime_schedule_source": "section_7_dag_list_schedule",
            "confidence_thresholds": CONFIDENCE_THRESHOLDS,
            "threshold_validation_table": THRESHOLD_VALIDATION_TABLE,
            "success_probability_table": SUCCESS_PROBABILITY_TABLE,
            "mean_timing_table": MEAN_TIMING_TABLE,
            "latency_timing_table": LATENCY_TIMING_TABLE,
            "mean_execution_time_ms_by_label": MEAN_EXECUTION_TIME_MS_BY_LABEL,
            "latency_execution_time_ms_by_label": LATENCY_EXECUTION_TIME_MS_BY_LABEL,
            "dag_optimal_order": list(DAG_OPTIMAL_ORDER_LABELS),
            "dag_worker_groups": [list(group) for group in DAG_OPTIMAL_RESULTS["worker_groups_labels"]],
            "dag_expected_duration_ms": DAG_OPTIMAL_RESULTS["expected_duration_ms"],
            "dag_success_probability": DAG_OPTIMAL_RESULTS["success_probability"],
            "dag_optimal_results": DAG_OPTIMAL_RESULTS,
            "exhaustive_verifier_result": EXHAUSTIVE_VERIFIER_RESULT,
            "runtime_schedule_results": RUNTIME_SCHEDULE_RESULTS,
            "runtime_cascade_order": list(RUNTIME_CASCADE_ORDER),
            "worker_model_groups": {worker_name: list(group) for worker_name, group in zip(WORKER_NAMES, WORKER_MODEL_GROUPS)},
            "actual_worker_groups_used": {worker_name: list(group) for worker_name, group in ACTIVE_WORKER_GROUP_BY_NAME.items()},
            "real_runtime_metrics": runtime_metrics,
            "final_prediction_count_by_model": {model_name: int(np.count_nonzero(chosen_models == model_name)) for model_name in MODELS},
            "success_count_by_model": {model_name: int(np.count_nonzero((chosen_models == model_name) & success_flags)) for model_name in MODELS},
            "idk_count_by_model": {model_name: int(idk_count_by_model[model_name]) for model_name in MODELS},
            "execution_count_by_model": {model_name: int(execution_count_by_model[model_name]) for model_name in MODELS},
            "total_execution_time_ms_by_model": total_execution_time_ms_by_model,
            "mean_execution_time_ms_by_model": mean_execution_time_ms_by_model,
        }
        return results, final_predictions, chosen_models, success_flags

    finally:
        for queue in worker_queues.values():
            queue.put(None)
        for process in processes:
            process.join()


real_system_results, real_final_predictions, real_chosen_models, real_success_flags = run_real_time_test(CAMERA_FPS)


## Print and Save Metrics

This cell prints the shared-MPS runtime metrics and saves the JSON results plus NPZ predictions when `SAVE_RESULTS` is True.


In [10]:
print("Shared-MPS runtime test of the selected schedule")
print("Actual worker groups used:", real_system_results["actual_worker_groups_used"])
print("Total samples:", real_system_results["total_samples"])
print("Success rate:", round(real_system_results["successful_classification_rate"], 4))
print("Accuracy:", round(real_system_results["accuracy"], 4))
print("Mean latency (ms):", round(real_system_results["mean_latency_ms"], 3))
print("P95 latency (ms):", round(real_system_results["p95_latency_ms"], 3))
print("Configured camera FPS cap:", real_system_results["configured_input_fps"])
print("Throughput (FPS):", round(real_system_results["throughput_fps"], 3))
print()
print("DAG optimal order:", real_system_results["dag_optimal_order"])
print("DAG worker groups:", real_system_results["dag_worker_groups"])
print("DAG expected duration (ms):", round(real_system_results["dag_expected_duration_ms"], 3))
print("DAG success probability:", round(real_system_results["dag_success_probability"], 6))
print()
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['execution_count_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['mean_execution_time_ms_by_model'][model_name]:.3f}")

if SAVE_RESULTS:
    RESULTS_PATH.write_text(json.dumps(real_system_results, indent=2) + "", encoding="utf-8")
    np.savez_compressed(
        PREDICTIONS_PATH,
        predictions=real_final_predictions,
        chosen_models=real_chosen_models,
        success_flags=real_success_flags,
    )
    print("Saved:", RESULTS_PATH.name)
    print("Saved:", PREDICTIONS_PATH.name)
else:
    print("SAVE_RESULTS is False; no files were written.")


Shared-MPS runtime test of the selected schedule
Actual worker groups used: {'mps-worker-0': ['resnet18'], 'mps-worker-1': ['resnet34'], 'mps-worker-2': ['resnet50'], 'mps-worker-3': ['resnet152']}
Total samples: 10000
Success rate: 0.5937
Accuracy: 0.552
Mean latency (ms): 45.236
P95 latency (ms): 77.886
Configured camera FPS cap: 60.0
Throughput (FPS): 13.638

DAG optimal order: ['A', 'B', 'C', 'D']
DAG worker groups: [['A'], ['B'], ['C'], ['D']]
DAG expected duration (ms): 14.471
DAG success probability: 0.58975

Final prediction count by model:
  resnet18: 3290
  resnet34: 1417
  resnet50: 141
  resnet152: 1089
Execution count by model:
  resnet18: 10000
  resnet34: 10000
  resnet50: 10000
  resnet152: 10000
Mean execution time by model (ms):
  resnet18: 15.958
  resnet34: 23.744
  resnet50: 36.996
  resnet152: 67.887
Saved: paper173_sec7_camera_fps_results.json
Saved: paper173_sec7_camera_fps_predictions.npz
